In [53]:
import keras
from keras.preprocessing import image
from keras.applications.inception_v3 import preprocess_input,decode_predictions, InceptionV3
import numpy as np
import tensorflow as tf

In [54]:
model = InceptionV3(include_top=True, weights='imagenet',input_tensor=None, input_shape=None)

In [55]:
def predict(image_file):
    img = image.load_img(image_file, target_size=(299, 299))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    preds = model.predict(x)
    decoded_preds = decode_predictions(preds, top=3)[0]
    return decoded_preds

In [56]:
from flask import Flask, current_app, request, jsonify
import io
import base64
import logging

In [57]:
app = Flask(__name__)

# Assume this is your actual ML model's predict function
# def predict(image): 
#     return {"class": "cat", "confidence": 0.99}

@app.route('/', methods=['POST'])
def predict_api():  # 1. Renamed to prevent colliding with your ML predict function
    data = {}
    try:
        data = request.get_json()['data']
    except Exception:
        return jsonify(status_code=400, message="Invalid request. Please provide a valid JSON payload with 'data' field."), 400

    data = base64.b64decode(data)
    image = io.BytesIO(data)
    
    # Now this correctly calls the external predict function, not itself
    predictions = predict(image) 
    
    current_app.logger.info(f"Predictions: {predictions}")
    return jsonify(predictions=predictions)

if __name__ == '__main__':
    # 2. Added use_reloader=False so it doesn't crash Jupyter
    app.run(host='0.0.0.0', port=8080, debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8080
 * Running on http://10.147.248.87:8080
Press CTRL+C to quit
127.0.0.1 - - [15/Sep/2026 20:36:42] "GET / HTTP/1.1" 405 -
127.0.0.1 - - [15/Sep/2026 20:36:42] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [15/Sep/2026 20:36:47] "GET / HTTP/1.1" 405 -
10.147.248.87 - - [15/Sep/2026 20:36:48] "GET / HTTP/1.1" 405 -
10.147.248.87 - - [15/Sep/2026 20:36:48] "GET /favicon.ico HTTP/1.1" 404 -
